In [ ]:

from __future__ import annotations

import re
import math
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix


# ============================================================
# 1) PERSPECTIVE CLASSIFIER (SAFE)
# ============================================================

FIRST_PERSON = re.compile(
    r"\b(i|me|my|mine|i've|i'm|i am|i feel|i have|i am experiencing)\b",
    re.IGNORECASE,
)

THIRD_PERSON = re.compile(
    r"\b("
    r"my (mother|father|husband|wife|daughter|son|child)|"
    r"he|she|his|her|they|their|"
    r"caregiver|caretaker|"
    r"mother reports|father reports|"
    r"patient's mother|patient's father|"
    r"caregiver reports|daughter reports|son reports"
    r")\b",
    re.IGNORECASE,
)

def classify_perspective(text: object) -> str:
    """
    Returns: first_person / third_person / mixed / unclear / unknown
    NOTE: Works on text in-memory; do not export text.
    """
    if not isinstance(text, str) or not text.strip():
        return "unknown"

    has_first = bool(FIRST_PERSON.search(text))
    has_third = bool(THIRD_PERSON.search(text))

    if has_first and not has_third:
        return "first_person"
    elif has_third and not has_first:
        return "third_person"
    elif has_first and has_third:
        return "mixed"
    else:
        return "unclear"


# ============================================================
# 2) BUILD msg_df and person_df (NO TEXT PASSTHROUGH)
# ============================================================

def build_msg_and_person_df(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Creates:
      - msg_df: message-level aggregated table (NO raw text)
      - person_df: person-level aggregated table
    """
    required = {"a_id", "m_id", "label", "count", "created_time", "dep_start", "cvd_start"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    msg_df = (
        df.groupby(["a_id", "m_id"], as_index=False)
          .agg(
              label=("label", "first"),
              count=("count", "max"),
              created_time=("created_time", "first"),
              dep_start=("dep_start", "first"),
              cvd_start=("cvd_start", "first"),
          )
    )

    # message-level positivity (baseline)
    msg_df["msg_pos"] = (msg_df["count"] >= 1).astype(int)

    person_df = (
        msg_df.groupby("a_id", as_index=False)
              .agg(
                  label=("label", "first"),
                  dep_start=("dep_start", "first"),
                  cvd_start=("cvd_start", "first"),
                  total_messages=("m_id", "nunique"),
                  positive_messages=("msg_pos", "sum"),
                  total_hits=("count", "sum"),
              )
    )

    # Ensure datetime types
    for c in ["created_time", "dep_start", "cvd_start"]:
        if c in msg_df.columns:
            msg_df[c] = pd.to_datetime(msg_df[c], errors="coerce")
        if c in person_df.columns:
            person_df[c] = pd.to_datetime(person_df[c], errors="coerce")

    return msg_df, person_df


# ============================================================
# 3) PERSON-LEVEL THRESHOLD METRICS
# ============================================================

def compute_metrics_at_threshold_person(
    df_person: pd.DataFrame,
    threshold: int,
    prevalence: float = 0.25,
    eps: float = 0.5
) -> Dict:
    eligible = df_person[df_person["total_messages"] >= threshold].copy()
    n_eligible = len(eligible)

    if n_eligible == 0:
        return {
            "threshold": int(threshold),
            "n_eligible": 0,
            "n_flagged": 0,
            "tp": 0, "fp": 0, "fn": 0, "tn": 0,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "ppv_adj": np.nan,
            "npv_adj": np.nan,
            "or": np.nan,
            "or_ci_low": np.nan,
            "or_ci_high": np.nan,
        }

    y_pred = (eligible["positive_messages"] >= threshold).astype(int)
    y_true = eligible["label"].astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    denom_ppv = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
    ppv_adj = (sensitivity * prevalence) / denom_ppv if denom_ppv > 0 else np.nan

    denom_npv = (1 - sensitivity) * prevalence + specificity * (1 - prevalence)
    npv_adj = (specificity * (1 - prevalence)) / denom_npv if denom_npv > 0 else np.nan

    tp_c, fp_c, fn_c, tn_c = tp + eps, fp + eps, fn + eps, tn + eps
    or_est = (tp_c * tn_c) / (fp_c * fn_c)

    se_log_or = math.sqrt(1/tp_c + 1/fp_c + 1/fn_c + 1/tn_c)
    ci_lower = math.exp(math.log(or_est) - 1.96 * se_log_or)
    ci_upper = math.exp(math.log(or_est) + 1.96 * se_log_or)

    return {
        "threshold": int(threshold),
        "n_eligible": int(n_eligible),
        "n_flagged": int(y_pred.sum()),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "sensitivity": float(sensitivity) if pd.notna(sensitivity) else np.nan,
        "specificity": float(specificity) if pd.notna(specificity) else np.nan,
        "ppv_adj": float(ppv_adj) if pd.notna(ppv_adj) else np.nan,
        "npv_adj": float(npv_adj) if pd.notna(npv_adj) else np.nan,
        "or": float(or_est),
        "or_ci_low": float(ci_lower),
        "or_ci_high": float(ci_upper),
    }


# ============================================================
# 4) MESSAGE-LEVEL FIXED METRICS (NO THRESHOLD SWEEP)
# ============================================================

def compute_metrics_message_fixed(
    msg_df: pd.DataFrame,
    prevalence: float = 0.25,
    eps: float = 0.5
) -> Dict:
    if len(msg_df) == 0:
        return {
            "n_messages": 0,
            "n_flagged": 0,
            "tp": 0, "fp": 0, "fn": 0, "tn": 0,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "ppv_adj": np.nan,
            "npv_adj": np.nan,
            "or": np.nan,
            "or_ci_low": np.nan,
            "or_ci_high": np.nan,
        }

    y_true = msg_df["label"].astype(int).values
    y_pred = msg_df["msg_pos"].astype(int).values

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    denom_ppv = sensitivity * prevalence + (1 - specificity) * (1 - prevalence)
    ppv_adj = (sensitivity * prevalence) / denom_ppv if denom_ppv > 0 else np.nan

    denom_npv = (1 - sensitivity) * prevalence + specificity * (1 - prevalence)
    npv_adj = (specificity * (1 - prevalence)) / denom_npv if denom_npv > 0 else np.nan

    tp_c, fp_c, fn_c, tn_c = tp + eps, fp + eps, fn + eps, tn + eps
    or_est = (tp_c * tn_c) / (fp_c * fn_c)

    se_log_or = math.sqrt(1/tp_c + 1/fp_c + 1/fn_c + 1/tn_c)
    ci_lower = math.exp(math.log(or_est) - 1.96 * se_log_or)
    ci_upper = math.exp(math.log(or_est) + 1.96 * se_log_or)

    return {
        "n_messages": int(len(msg_df)),
        "n_flagged": int(y_pred.sum()),
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "sensitivity": float(sensitivity) if pd.notna(sensitivity) else np.nan,
        "specificity": float(specificity) if pd.notna(specificity) else np.nan,
        "ppv_adj": float(ppv_adj) if pd.notna(ppv_adj) else np.nan,
        "npv_adj": float(npv_adj) if pd.notna(npv_adj) else np.nan,
        "or": float(or_est),
        "or_ci_low": float(ci_lower),
        "or_ci_high": float(ci_upper),
    }


# ============================================================
# 5) TEMPORAL BLOCKS + RUNNER (EXPORTS ARE AGGREGATE-ONLY)
# ============================================================

@dataclass(frozen=True)
class TimeBlock:
    name: str
    start: str  # inclusive
    end: str    # inclusive

TIME_BLOCKS: List[TimeBlock] = [
    TimeBlock("drop_block_1", "2014-07-01", "2016-12-31"),
    TimeBlock("drop_block_2", "2017-01-01", "2019-07-31"),
    TimeBlock("drop_block_3", "2019-08-01", "2021-12-31"),
    TimeBlock("drop_block_4", "2022-01-01", "2024-07-31"),
]

def temporal_drop_perf_person_curve_and_msg_fixed(
    df_raw: pd.DataFrame,
    prevalence: float = 0.25,
    date_col_person: str = "dep_start",
    blocks: List[TimeBlock] = TIME_BLOCKS,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Drops persons whose dep_start is within each block.
    Outputs contain NO IDs and NO raw text.
    """
    msg_df, person_df = build_msg_and_person_df(df_raw)

    person_perf_rows: List[Dict] = []
    msg_perf_rows: List[Dict] = []
    run_summaries: List[Dict] = []

    for blk in blocks:
        start_dt = pd.to_datetime(blk.start)
        end_dt = pd.to_datetime(blk.end)

        in_block = person_df[date_col_person].between(start_dt, end_dt, inclusive="both")
        dropped_n = int(in_block.sum())

        person_keep = person_df.loc[~in_block].copy()
        msg_keep = msg_df[msg_df["a_id"].isin(person_keep["a_id"])].copy()

        # --- summaries (aggregate only)
        run_summaries.append({
            "run": blk.name,
            "dropped_start": blk.start,
            "dropped_end": blk.end,
            "person_full": int(len(person_df)),
            "person_dropped": int(dropped_n),
            "person_remaining": int(len(person_keep)),
            "person_pos_rate_remaining": float(person_keep["label"].astype(int).mean()) if len(person_keep) else np.nan,
            "messages_full": int(len(msg_df)),
            "messages_remaining": int(len(msg_keep)),
            "msg_pos_rate_remaining": float(msg_keep["label"].astype(int).mean()) if len(msg_keep) else np.nan,
        })

        # --- person-level curve
        max_t_pat = int(person_keep["total_messages"].max()) if len(person_keep) else 0
        for t in range(1, max_t_pat + 1):
            m = compute_metrics_at_threshold_person(person_keep, t, prevalence=prevalence)
            m.update({
                "run": blk.name,
                "date_drop_start": blk.start,
                "date_drop_end": blk.end,
                "level": "person",
            })
            person_perf_rows.append(m)

        # --- message-level fixed metrics
        mm = compute_metrics_message_fixed(msg_keep, prevalence=prevalence)
        mm.update({
            "run": blk.name,
            "date_drop_start": blk.start,
            "date_drop_end": blk.end,
            "level": "message",
        })
        msg_perf_rows.append(mm)

    person_perf_long = pd.DataFrame(person_perf_rows)
    msg_perf_by_run = pd.DataFrame(msg_perf_rows)
    run_summary = pd.DataFrame(run_summaries)

    # Remove any accidental ID/date columns before exporting
    person_perf_long = person_perf_long.drop(columns=[c for c in ["a_id", "m_id"] if c in person_perf_long.columns], errors="ignore")
    msg_perf_by_run = msg_perf_by_run.drop(columns=[c for c in ["a_id", "m_id"] if c in msg_perf_by_run.columns], errors="ignore")
    run_summary = run_summary.drop(columns=[c for c in ["a_id", "m_id"] if c in run_summary.columns], errors="ignore")

    return person_perf_long, msg_perf_by_run, run_summary


# ============================================================
# 6) CONCEPT SENSITIVITY (PUBLIC-SAFE)
#    - DOES NOT EXPORT TEXT
# ============================================================

SYMPTOM_LEXICON = {
    "depressed_mood": ["sad", "depressed", "down", "low mood", "tearful"],
    "anhedonia": ["no interest", "lost interest", "no pleasure", "nothing enjoyable"],
    "hopelessness": ["hopeless", "helpless", "nothing will get better"],
    "worthlessness_guilt": ["worthless", "guilty", "blame myself", "failure"],
    "suicidal_ideation": ["suicidal", "kill myself", "end my life", "not want to live"],
    "sleep_disturbance": ["can't sleep", "insomnia", "sleeping too much", "poor sleep"],
    "fatigue_low_energy": ["fatigue", "tired all the time", "no energy", "exhausted"],
    "concentration": ["can't concentrate", "trouble focusing", "poor concentration"],

    # --- SOMATIC ---
    "pain": ["pain", "ache", "sore"],
    "gi_symptoms": ["nausea", "constipation", "diarrhea", "stomach pain"],
    "headache_dizziness": ["headache", "dizzy", "lightheaded"],
    "cardiorespiratory": ["shortness of breath", "chest pain", "palpitations"],
}

PSYCHIATRIC = {
    "depressed_mood","anhedonia","hopelessness","worthlessness_guilt",
    "suicidal_ideation","sleep_disturbance","fatigue_low_energy","concentration"
}

def extract_symptoms(text: object) -> List[str]:
    if not isinstance(text, str):
        return []
    t = text.lower()
    out: List[str] = []
    for symptom, terms in SYMPTOM_LEXICON.items():
        for phrase in terms:
            if re.search(r"\b" + re.escape(phrase) + r"\b", t):
                out.append(symptom)
                break
    return out

def domain(symptoms: List[str]) -> str:
    has_psych = any(s in PSYCHIATRIC for s in symptoms)
    has_somatic = any(s not in PSYCHIATRIC for s in symptoms)
    if has_psych and not has_somatic:
        return "psychiatric"
    if has_somatic and not has_psych:
        return "somatic"
    if has_psych and has_somatic:
        return "mixed"
    return "other"

def concept_sensitivity_summary(
    df_raw: pd.DataFrame,
    threshold: int = 9
) -> pd.DataFrame:
    """
    Computes domain-based ablation screen rates.
    Returns an AGGREGATE summary table only (no IDs, no text).
    """
    required = {"a_id", "m_id", "label", "count", "msg_txt"}
    missing = sorted(required - set(df_raw.columns))
    if missing:
        raise KeyError(f"Missing required columns for concept sensitivity: {missing}")

    # Build msg_df with msg_txt kept ONLY IN-MEMORY (not returned/exported)
    msg_df = (
        df_raw.groupby(["a_id", "m_id"], as_index=False)
              .agg(
                  label=("label", "first"),
                  count=("count", "max"),
                  msg_txt=("msg_txt", "first"),
              )
    )
    msg_df["msg_pos"] = (msg_df["count"] >= 1).astype(int)

    pos_msgs = msg_df[msg_df["msg_pos"] == 1].copy()
    # In-memory text processing only
    pos_msgs["symptom_categories"] = pos_msgs["msg_txt"].apply(extract_symptoms)
    pos_msgs["domain"] = pos_msgs["symptom_categories"].apply(domain)

    # Person-level domain counts
    person_domain_counts = (
        pos_msgs.groupby(["a_id", "domain"])
               .size()
               .unstack(fill_value=0)
               .reset_index()
    )

    person_domain_counts["pos_all"] = (
        person_domain_counts.get("psychiatric", 0)
        + person_domain_counts.get("somatic", 0)
        + person_domain_counts.get("mixed", 0)
    )
    person_domain_counts["pos_no_psych"] = person_domain_counts.get("somatic", 0)
    person_domain_counts["pos_no_somatic"] = person_domain_counts.get("psychiatric", 0)

    # Screening indicators (kept private at person-level; we aggregate below)
    person_domain_counts["screen_all"] = (person_domain_counts["pos_all"] >= threshold).astype(int)
    person_domain_counts["screen_no_psych"] = (person_domain_counts["pos_no_psych"] >= threshold).astype(int)
    person_domain_counts["screen_no_somatic"] = (person_domain_counts["pos_no_somatic"] >= threshold).astype(int)

    # Aggregate-only summary
    n_all = int(person_domain_counts["screen_all"].sum())
    n_no_psych = int(person_domain_counts["screen_no_psych"].sum())
    n_no_somatic = int(person_domain_counts["screen_no_somatic"].sum())

    drop_psych = n_all - n_no_psych
    drop_somatic = n_all - n_no_somatic

    n_only_somatic_pos = int(((person_domain_counts.get("psychiatric", 0) == 0) &
                              (person_domain_counts.get("somatic", 0) > 0)).sum())
    n_only_psych_pos = int(((person_domain_counts.get("somatic", 0) == 0) &
                            (person_domain_counts.get("psychiatric", 0) > 0)).sum())

    # Do NOT return IDs
    summary = pd.DataFrame([{
        "threshold": int(threshold),
        "n_screen_all": n_all,
        "n_screen_no_psych": n_no_psych,
        "n_screen_no_somatic": n_no_somatic,
        "drop_psych": int(drop_psych),
        "drop_somatic": int(drop_somatic),
        "n_person_only_somatic_pos": n_only_somatic_pos,
        "n_person_only_psych_pos": n_only_psych_pos,
        "n_person_with_any_pos_msg": int(person_domain_counts.shape[0]),
    }])

    return summary
